# 🎓 大模型复杂推理后训练与强化对齐实战手册 (Google Colab / AWS 专版)

本 Notebook 专为**第一次上手大模型科研实践**设计。您只需在 Google Colab 中将运行时切换为 **T4 GPU**（菜单栏：`代码执行程序` -> `更改运行时类型` -> `T4 GPU`），或部署在 AWS `g4dn.xlarge` / `g5.xlarge` 实例上，即可按顺序逐格运行。

### 本实验涵盖的三大科研阶段：
1. **Stage 1 — 监督微调 (SFT with QLoRA / DoRA)**：回答 **RQ1**（权重分解 DoRA 与全线性层注入对长思维链推理能力的影响）。
2. **Stage 2 — 离线偏好对齐 (DPO vs. SimPO)**：回答 **RQ2**（无参考模型且长度归一化的 SimPO 如何抑制冗余废话并节省显存）。
3. **Stage 3 — 在线强化学习 (GRPO, DeepSeek-R1 范式)**：回答 **RQ3**（可验证规则奖励如何引导小模型学会结构化思考 `<think>...</think>` 与自我纠错）。

## Step 0: 检查 GPU 算力环境并安装依赖

In [ ]:
!nvidia-smi
!pip install -q torch transformers datasets accelerate peft trl bitsandbytes pyyaml matplotlib seaborn

## Step 1: 先跑通 30 秒冒烟测试（验证所有模块与公式无误）

In [ ]:
!python run_pipeline.py --mode smoke_test

## Step 2 (RQ1): 运行 Stage 1 监督微调 (LoRA vs. DoRA 消融对比)
先用 `--max_steps 100` 快速观察 Loss 收敛曲线（约 5~8 分钟），确认无误后可去掉 `--max_steps` 跑完整 Epoch。

In [ ]:
# 1. 标准 QLoRA (r=16, all_linear)
!python run_pipeline.py --mode sft --config configs/sft_lora.yaml --lora_r 16 --target_scope all_linear --max_steps 100

# 2. 权重分解 DoRA (r=16, all_linear)
!python run_pipeline.py --mode sft --config configs/sft_lora.yaml --use_dora --lora_r 16 --target_scope all_linear --max_steps 100

## Step 3 (RQ2): 运行 Stage 2 偏好对齐 (DPO vs. SimPO)
对比有参考模型的 DPO 与无参考模型长度归一化的 SimPO 在隐式奖励准确率 (Reward Accuracy)、显存占用和输出 Token 长度上的差异。

In [ ]:
# 1. 运行 DPO
!python run_pipeline.py --mode align --config configs/align_dpo_simpo.yaml --algo dpo --max_steps 60

# 2. 运行 SimPO
!python run_pipeline.py --mode align --config configs/align_dpo_simpo.yaml --algo simpo --max_steps 60

## Step 4 (RQ3): 运行 Stage 3 群组相对策略优化 (GRPO)
对每个数学问题在线采样 $G=4$ 条轨迹，利用**数值正确性 + `<think>` 标签格式遵循度**计算规则奖励，观察组内相对优势更新。

In [ ]:
!python run_pipeline.py --mode grpo --config configs/rl_grpo.yaml --max_steps 50

## Step 5: 域内 (GSM8K) 与域外 (SVAMP) 评测及论文图表绘制

In [ ]:
!python run_pipeline.py --mode eval --benchmark gsm8k --adapter_path outputs/stage3_grpo/final_grpo_adapter
!python run_pipeline.py --mode eval --benchmark svamp --adapter_path outputs/stage3_grpo/final_grpo_adapter
!python run_pipeline.py --mode plot